<a href="https://colab.research.google.com/github/jetendarsoothar-png/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jetendarsoothar-png/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

I will prioritize pages for content refresh using two signals: content age and search volume.

Older content gets a higher refresh priority because stale content may need updating.
Pages with stronger recent search volume get higher priority because refreshing them may have more potential impact.

### Reason codes

- STALE_HIGH_VOLUME — old content with strong recent search volume
- STALE_LOW_VOLUME — old content with weaker recent search volume
- FRESH_HIGH_VOLUME — newer content with strong recent search volume
- FRESH_LOW_VOLUME — newer content with weaker recent search volume

The score is decision-support only. It is not a prediction of future traffic or business impact.

In [14]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current folder:
/content

Files/folders here:
['.config', 'flyrank-ml-internship', 'sample_data']


In [15]:
!git clone https://github.com/jetendarsoothar-png/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [16]:
import os

print(os.listdir("/content/flyrank-ml-internship"))

['submission', 'DATA_USE.md', 'notebooks', '.github', 'outputs', 'GUIDE.md', 'requirements.txt', 'CLAUDE.md', 'docs', 'scripts', 'README.md', 'skills', 'data', 'SETUP.md', 'AGENTS.md', '.gitignore', 'LICENSE', 'work', '.git']


In [17]:
import os

data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(data_path))

if os.path.exists(data_path):
    print("Dataset found ✅")
else:
    print("Dataset not found ❌")
    print("\nRaw folder contents:")
    print(os.listdir("/content/flyrank-ml-internship/data/raw"))

Dataset exists: True
Dataset found ✅


In [18]:
import pandas as pd
import numpy as np

# Load FlyRank dataset
data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# -----------------------------
# Signal 1: Content age
# -----------------------------

df["content_age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 180, 365, 730, np.inf],
    labels=["0-180", "181-365", "366-730", "731+"]
)

age_check = (
    df.groupby("content_age_bucket", observed=False)
      .agg(
          n=("content_age_days", "size"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

print("\nSignal 1 — Content Age")
print(age_check)


# -----------------------------
# Signal 2: Search volume
# -----------------------------

df["impressions_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

volume_check = (
    df.groupby("impressions_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          avg_content_age=("content_age_days", "mean")
      )
      .reset_index()
)

print("\nSignal 2 — Impressions / Volume")
print(volume_check)

Rows: 30000
Columns: 44

Signal 1 — Content Age
  content_age_bucket      n  avg_impressions
0              0-180  12272      5025.862451
1            181-365  11368      5398.772871
2            366-730   6360      5182.445755
3               731+      0              NaN

Signal 2 — Impressions / Volume
  impressions_bucket      n  avg_content_age
0                Low  10003       246.243327
1             Medium   9997       268.427428
2               High  10000       253.839300


### Signal verdicts

**Content age — MIXED:** Older content is not consistently associated with higher impressions. The 181–365 day bucket has the highest average impressions, while the 366–730 day bucket is lower. The 731+ bucket has no observations, so the strongest staleness category cannot be evaluated.

**Impressions / volume — MIXED:** Average content age is very similar across low, medium, and high impression buckets. Impressions provide a useful volume signal, but this check does not show a strong relationship between volume and content age.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import os

output_dir = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

print("Output folder ready:", os.path.exists(output_dir))

Output folder ready: True


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Build the ranked queue

import numpy as np
import pandas as pd

queue = df.copy()

# -----------------------------
# 1. Create score components
# -----------------------------

# Older content gets more points.
# We use the observed age range rather than inventing a 731+ category.
age_score = pd.cut(
    queue["content_age_days"],
    bins=[-1, 180, 365, np.inf],
    labels=[1, 2, 3]
).astype(int)

# Higher impressions = more potential search volume.
# Rank into three equal-sized groups.
volume_score = pd.qcut(
    queue["impressions_90d"],
    q=3,
    labels=[1, 2, 3],
    duplicates="drop"
).astype(int)

# Combined baseline score
queue["baseline_score"] = age_score + volume_score


# -----------------------------
# 2. Reason code
# -----------------------------

queue["reason_code"] = np.select(
    [
        (age_score == 3) & (volume_score == 3),
        (age_score == 3) & (volume_score < 3),
        (age_score < 3) & (volume_score == 3)
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE_LOWER_VOLUME",
        "FRESH_HIGH_VOLUME"
    ],
    default="LOWER_PRIORITY"
)


# -----------------------------
# 3. Action label
# -----------------------------

queue["action"] = np.select(
    [
        queue["baseline_score"] >= 6,
        queue["baseline_score"] >= 4
    ],
    [
        "REFRESH_NOW",
        "REVIEW"
    ],
    default="MONITOR"
)


# -----------------------------
# 4. Rank the queue
# -----------------------------

queue = queue.sort_values(
    ["baseline_score", "impressions_90d", "content_age_days"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)


# -----------------------------
# 5. Select output columns
# -----------------------------

output_cols = [
    "rank",
    "baseline_score",
    "reason_code",
    "action",
    "content_age_days",
    "impressions_90d"
]

baseline_queue = queue[output_cols].copy()


# -----------------------------
# 6. Write required CSV
# -----------------------------

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("Baseline queue created successfully.")
print("Rows:", len(baseline_queue))
print("Saved to:", output_path)

print("\nAction counts:")
print(baseline_queue["action"].value_counts())

print("\nTop 10:")
print(baseline_queue.head(10).to_string(index=False))

Baseline queue created successfully.
Rows: 30000
Saved to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Action counts:
action
REVIEW         15396
MONITOR        12581
REFRESH_NOW     2023
Name: count, dtype: int64

Top 10:
 rank  baseline_score       reason_code      action  content_age_days  impressions_90d
    1               6 STALE_HIGH_VOLUME REFRESH_NOW               537           517715
    2               6 STALE_HIGH_VOLUME REFRESH_NOW               445           517109
    3               6 STALE_HIGH_VOLUME REFRESH_NOW               445           509252
    4               6 STALE_HIGH_VOLUME REFRESH_NOW               445           463103
    5               6 STALE_HIGH_VOLUME REFRESH_NOW               482           416180
    6               6 STALE_HIGH_VOLUME REFRESH_NOW               445           345111
    7               6 STALE_HIGH_VOLUME REFRESH_NOW               445           309192
    8               6 STALE_HIGH_VOLUME REFRESH_NOW   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 review

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["baseline_score"] == 6,
    "High baseline score because both age and volume are high.",
    "Moderate baseline score based on age and/or volume."
)

top20["what_would_make_it_wrong"] = np.where(
    top20["action"] == "REFRESH_NOW",
    "The page may not need a refresh if its current content is still accurate, relevant, and performing well.",
    "The signal thresholds may not reflect the page's true refresh need."
)

print("Top-20 review:")
print(
    top20[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)


Top-20 review:
 rank  baseline_score       reason_code      action                                           confidence_note                                                                                 what_would_make_it_wrong
    1               6 STALE_HIGH_VOLUME REFRESH_NOW High baseline score because both age and volume are high. The page may not need a refresh if its current content is still accurate, relevant, and performing well.
    2               6 STALE_HIGH_VOLUME REFRESH_NOW High baseline score because both age and volume are high. The page may not need a refresh if its current content is still accurate, relevant, and performing well.
    3               6 STALE_HIGH_VOLUME REFRESH_NOW High baseline score because both age and volume are high. The page may not need a refresh if its current content is still accurate, relevant, and performing well.
    4               6 STALE_HIGH_VOLUME REFRESH_NOW High baseline score because both age and volume are high. The page may no

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks and leakage check

The baseline produces a strong concentration of similar high-score pages at the top of the queue. This is a limitation because the rule uses only two signals, so pages with similar age and volume can receive the same priority.

A top pick could be wrong if the page is already accurate, relevant, and performing well despite being old and high-volume. The rule should therefore be treated as decision-support rather than a prediction.

The baseline uses content_age_days and impressions_90d only. Known label-derived fields such as trend_direction and trend_pct are not used, and no future-window inputs are used.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

print("=== Weak Pick Check ===")

# Show the lowest-confidence boundary around the top-20 cutoff
boundary = baseline_queue.iloc[20:30][
    [
        "rank",
        "baseline_score",
        "reason_code",
        "action",
        "content_age_days",
        "impressions_90d"
    ]
]

print("\nRows immediately below the Top-20:")
print(boundary.to_string(index=False))


# -----------------------------
# Leakage check
# -----------------------------

print("\n=== Leakage Check ===")

# Inputs actually used by the rule
used_inputs = [
    "content_age_days",
    "impressions_90d"
]

print("Inputs used by baseline:", used_inputs)

# Check that known label-derived / future-looking fields
# are not used in the scoring calculation.
for col in ["trend_direction", "trend_pct"]:
    if col in queue.columns:
        print(f"{col}: present in dataset, but NOT used by baseline.")

# Search for obviously future-window fields
future_terms = [
    "future",
    "next_",
    "after_",
    "post_",
    "30d_future",
    "90d_future"
]

future_like_columns = [
    col for col in queue.columns
    if any(term in col.lower() for term in future_terms)
]

print("\nFuture-looking column names found:")
print(future_like_columns if future_like_columns else "None found")

print("\nLeakage check: PASS")
print("The baseline score uses only content_age_days and impressions_90d.")

=== Weak Pick Check ===

Rows immediately below the Top-20:
 rank  baseline_score       reason_code      action  content_age_days  impressions_90d
   21               6 STALE_HIGH_VOLUME REFRESH_NOW               480           125081
   22               6 STALE_HIGH_VOLUME REFRESH_NOW               390           125049
   23               6 STALE_HIGH_VOLUME REFRESH_NOW               417           124870
   24               6 STALE_HIGH_VOLUME REFRESH_NOW               421           123561
   25               6 STALE_HIGH_VOLUME REFRESH_NOW               466           122288
   26               6 STALE_HIGH_VOLUME REFRESH_NOW               463           121045
   27               6 STALE_HIGH_VOLUME REFRESH_NOW               421           118878
   28               6 STALE_HIGH_VOLUME REFRESH_NOW               480           117983
   29               6 STALE_HIGH_VOLUME REFRESH_NOW               421           117741
   30               6 STALE_HIGH_VOLUME REFRESH_NOW               480 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.